# GPT-2 Transformer Block Distillation into Neural Cellular Automata

This notebook demonstrates how to distill a single GPT-2 transformer block into a Neural Cellular Automata (NCA). The key idea is to trade a large number of parameters (~7M for a transformer block) for iterative compute steps using a much smaller NCA (~200K-500K parameters).

The NCA operates on a 2D grid where:
- Rows = Sequence positions
- Columns = Hidden dimension features

By running the NCA for many steps, local update rules learn to approximate the global computations (attention + MLP) of the transformer block.

## Installation

You will need Python 3.11 or later, and a working JAX installation. For example, you can install JAX with:

In [ ]:
%pip install -U "jax[cuda]"

Then, install CAX and additional dependencies:

In [ ]:
%pip install -U "cax[examples]" transformers torch datasets

## Imports

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from flax import nnx
from tqdm.auto import tqdm

from cax.nn.pool import Pool

# Local imports (from this example directory)
from data import get_or_create_activations
from nca import GPT2BlockNCA, count_params

## Configuration

In [ ]:
# Random seed
seed = 0

# GPT-2 configuration
block_idx = 0  # Which transformer block to distill (0-11)
hidden_dim = 768  # GPT-2 small hidden dimension
seq_len = 64  # Sequence length for training

# NCA architecture - key: compress 768 -> 64 for tractable NCA
compressed_dim = 64  # NCA operates on this smaller grid
channel_size = 16  # NCA state channels
perception_size = 64  # Perception vector size
hidden_layer_sizes = (128,)  # Update MLP hidden layers (single layer)
local_kernel_size = (3, 3)  # Local perception kernel
row_kernel_size = 5  # Feature mixing kernel
col_kernel_size = 5  # Sequence mixing kernel
step_size = 0.1  # Residual step size
cell_dropout_rate = 0.1  # Dropout rate

# Training configuration
num_nca_steps = 32  # NCA steps per forward pass
pool_size = 256  # Training pool size
batch_size = 8  # Batch size
learning_rate = 1e-3  # Learning rate
num_train_steps = 4096  # Total training steps
num_data_samples = 1024  # Number of text samples for activation dataset

# Initialize random keys
key = jax.random.key(seed)
rngs = nnx.Rngs(seed)

## Collect GPT-2 Activations

Activations are cached to disk after first run, so subsequent runs are instant.

In [ ]:
# Get activations (uses GPU for speed, caches to disk)
# First run: ~1-2 minutes on GPU, ~10-15 minutes on CPU
# Subsequent runs: instant (loaded from cache)
dataset = get_or_create_activations(
    block_idx=block_idx,
    num_samples=num_data_samples,
    max_length=seq_len,
    batch_size=64,  # Large batch for GPU speed
    device=None,  # Auto-selects GPU if available
)

print(f"\nDataset shapes:")
print(f"  Input:  {dataset['input'].shape}")
print(f"  Output: {dataset['output'].shape}")
print(f"  Mask:   {dataset['mask'].shape}")

In [ ]:
# Split into train/test
split_idx = int(0.9 * len(dataset['input']))

train_dataset = {
    'input': dataset['input'][:split_idx],
    'output': dataset['output'][:split_idx],
    'mask': dataset['mask'][:split_idx],
}

test_dataset = {
    'input': dataset['input'][split_idx:],
    'output': dataset['output'][split_idx:],
    'mask': dataset['mask'][split_idx:],
}

print(f"Train samples: {len(train_dataset['input'])}")
print(f"Test samples: {len(test_dataset['input'])}")

## Instantiate NCA

In [ ]:
# Create NCA model with compression (768 -> 64 -> NCA -> 64 -> 768)
nca = GPT2BlockNCA(
    hidden_dim=hidden_dim,
    compressed_dim=compressed_dim,
    channel_size=channel_size,
    perception_size=perception_size,
    hidden_layer_sizes=hidden_layer_sizes,
    local_kernel_size=local_kernel_size,
    row_kernel_size=row_kernel_size,
    col_kernel_size=col_kernel_size,
    step_size=step_size,
    cell_dropout_rate=cell_dropout_rate,
    rngs=rngs,
)

nca_params = count_params(nca)
gpt2_block_params = 7_077_888  # Approximate for GPT-2 small block

print(f"NCA parameters: {nca_params:,}")
print(f"GPT-2 block parameters: {gpt2_block_params:,}")
print(f"Compression ratio: {gpt2_block_params / nca_params:.1f}x")
print(f"\nNCA grid size: {seq_len} x {compressed_dim} (vs {seq_len} x {hidden_dim} uncompressed)")

## Loss Functions

In [ ]:
def mse_loss(pred, target, mask):
    """Masked MSE loss."""
    # Expand mask for hidden dimension
    mask_expanded = mask[..., None]  # (batch, seq, 1)
    
    squared_error = jnp.square(pred - target)
    masked_error = squared_error * mask_expanded
    
    return jnp.sum(masked_error) / (jnp.sum(mask_expanded) + 1e-8)


def cosine_similarity_loss(pred, target, mask):
    """Masked cosine similarity loss (1 - cosine_sim)."""
    mask_expanded = mask[..., None]
    
    pred_norm = pred / (jnp.linalg.norm(pred, axis=-1, keepdims=True) + 1e-8)
    target_norm = target / (jnp.linalg.norm(target, axis=-1, keepdims=True) + 1e-8)
    
    cosine_sim = jnp.sum(pred_norm * target_norm, axis=-1, keepdims=True)
    loss = (1 - cosine_sim) * mask_expanded
    
    return jnp.sum(loss) / (jnp.sum(mask_expanded) + 1e-8)

## Initialize Pool

In [ ]:
key, init_key = jax.random.split(key)

# Sample random indices for initial pool
init_indices = jax.random.randint(init_key, (pool_size,), 0, len(train_dataset['input']))

# Initialize states from input activations
init_inputs = train_dataset['input'][init_indices]
init_states = jax.vmap(nca.init_state)(init_inputs)

pool = Pool.create({
    'state': init_states,
    'data_idx': init_indices,
})

print(f"Pool initialized with {pool_size} states")
print(f"State shape: {init_states.shape}")

## Optimizer

In [ ]:
# Learning rate schedule with warmup and cosine decay
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=learning_rate * 0.1,
    peak_value=learning_rate,
    warmup_steps=500,
    decay_steps=num_train_steps,
    end_value=learning_rate * 0.01,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
)

# wrt=nnx.Param specifies which parameters to optimize (required in Flax 0.11+)
optimizer = nnx.Optimizer(nca, optimizer, wrt=nnx.Param)

## Training Step

In [ ]:
@nnx.jit
def loss_fn(nca, state, target, mask):
    """Compute loss for a batch of states."""
    # Run NCA for num_nca_steps
    state_axes = nnx.StateAxes({nnx.RngState: 0, ...: None})
    final_state = nnx.split_rngs(splits=state.shape[0])(
        nnx.vmap(
            lambda nca, s: nca(s, num_steps=num_nca_steps),
            in_axes=(state_axes, 0),
        )
    )(nca, state)
    
    # Extract output activations
    pred = nca.extract_output(final_state)
    
    # Compute losses
    mse = mse_loss(pred, target, mask)
    cosine = cosine_similarity_loss(pred, target, mask)
    
    total_loss = mse + 0.1 * cosine
    
    return total_loss, {'mse': mse, 'cosine': cosine, 'final_state': final_state}


@nnx.jit
def train_step(nca, optimizer, pool, train_data, key):
    """Single training step."""
    sample_key, new_sample_key = jax.random.split(key)
    
    # Sample from pool
    pool_idx, batch = pool.sample(sample_key, batch_size=batch_size)
    state = batch['state']
    data_idx = batch['data_idx']
    
    # Get corresponding target data
    target = train_data['output'][data_idx]
    mask = train_data['mask'][data_idx]
    
    # Compute loss and gradients
    (loss, aux), grad = nnx.value_and_grad(loss_fn, has_aux=True)(nca, state, target, mask)
    optimizer.update(nca, grad)
    
    final_state = aux['final_state']
    
    # Compute per-sample losses for sorting
    def single_mse(s, t, m):
        pred = nca.extract_output(s[None])[0]
        return mse_loss(pred[None], t[None], m[None])
    
    losses = jax.vmap(single_mse)(final_state, target, mask)
    
    # Sort by loss (descending) and replace worst with fresh sample
    sort_idx = jnp.argsort(losses, descending=True)
    pool_idx = pool_idx[sort_idx]
    final_state = final_state[sort_idx]
    data_idx = data_idx[sort_idx]
    
    # Replace worst sample with fresh initialization
    new_data_idx = jax.random.randint(new_sample_key, (), 0, len(train_data['input']))
    new_input = train_data['input'][new_data_idx]
    new_state = nca.init_state(new_input)
    
    final_state = final_state.at[0].set(new_state)
    data_idx = data_idx.at[0].set(new_data_idx)
    
    # Update pool
    pool = pool.update(pool_idx, {'state': final_state, 'data_idx': data_idx})
    
    return loss, aux['mse'], aux['cosine'], pool

## Training Loop

In [ ]:
print_interval = 100

pbar = tqdm(range(num_train_steps), desc="Training", unit="step")
losses = []
mse_losses = []
cosine_losses = []

for i in pbar:
    key, step_key = jax.random.split(key)
    loss, mse, cosine, pool = train_step(nca, optimizer, pool, train_dataset, step_key)
    
    losses.append(float(loss))
    mse_losses.append(float(mse))
    cosine_losses.append(float(cosine))
    
    if i % print_interval == 0 or i == num_train_steps - 1:
        avg_loss = sum(losses[-print_interval:]) / len(losses[-print_interval:])
        avg_mse = sum(mse_losses[-print_interval:]) / len(mse_losses[-print_interval:])
        pbar.set_postfix({
            'loss': f'{avg_loss:.4e}',
            'mse': f'{avg_mse:.4e}',
        })

## Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Smooth the losses for visualization
window = 50
smooth_losses = [sum(losses[max(0,i-window):i+1])/len(losses[max(0,i-window):i+1]) for i in range(len(losses))]
smooth_mse = [sum(mse_losses[max(0,i-window):i+1])/len(mse_losses[max(0,i-window):i+1]) for i in range(len(mse_losses))]

axes[0].plot(smooth_losses)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Total Loss')
axes[0].set_title('Training Loss')
axes[0].set_yscale('log')

axes[1].plot(smooth_mse)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('MSE Loss')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## Evaluate on Test Set

In [ ]:
def evaluate(nca, test_data, num_steps, eval_batch_size=32):
    """Evaluate NCA on test data in batches to save memory."""
    num_samples = len(test_data['input'])
    all_preds = []
    all_final_states = []
    
    # Process in batches
    for i in range(0, num_samples, eval_batch_size):
        batch_input = test_data['input'][i:i+eval_batch_size]
        batch_size = len(batch_input)
        
        # Initialize states from test inputs
        states = jax.vmap(nca.init_state)(batch_input)
        
        # Run NCA
        state_axes = nnx.StateAxes({nnx.RngState: 0, ...: None})
        final_states = nnx.split_rngs(splits=batch_size)(
            nnx.vmap(
                lambda nca, s: nca(s, num_steps=num_steps),
                in_axes=(state_axes, 0),
            )
        )(nca, states)
        
        # Extract predictions
        pred = nca.extract_output(final_states)
        all_preds.append(pred)
        all_final_states.append(final_states)
    
    # Concatenate results
    pred = jnp.concatenate(all_preds, axis=0)
    final_states = jnp.concatenate(all_final_states, axis=0)
    target = test_data['output']
    mask = test_data['mask']
    
    # Compute metrics
    mse = mse_loss(pred, target, mask)
    cosine = cosine_similarity_loss(pred, target, mask)
    
    # Cosine similarity (positive)
    mask_expanded = mask[..., None]
    pred_norm = pred / (jnp.linalg.norm(pred, axis=-1, keepdims=True) + 1e-8)
    target_norm = target / (jnp.linalg.norm(target, axis=-1, keepdims=True) + 1e-8)
    cosine_sim = jnp.sum(pred_norm * target_norm, axis=-1, keepdims=True)
    avg_cosine_sim = jnp.sum(cosine_sim * mask_expanded) / jnp.sum(mask_expanded)
    
    return {
        'mse': float(mse),
        'cosine_loss': float(cosine),
        'cosine_similarity': float(avg_cosine_sim),
        'pred': pred,
        'final_states': final_states,
    }

# Evaluate
results = evaluate(nca, test_dataset, num_nca_steps)

print(f"Test Results ({num_nca_steps} NCA steps):")
print(f"  MSE Loss: {results['mse']:.6f}")
print(f"  Cosine Similarity: {results['cosine_similarity']:.4f}")

## Visualize Activations

In [ ]:
# Pick a test sample to visualize
sample_idx = 0

input_act = test_dataset['input'][sample_idx]
target_act = test_dataset['output'][sample_idx]
pred_act = results['pred'][sample_idx]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Input activation
im0 = axes[0, 0].imshow(input_act[:64, :64], aspect='auto', cmap='RdBu')
axes[0, 0].set_title('Input Activation (first 64x64)')
axes[0, 0].set_xlabel('Hidden Dim')
axes[0, 0].set_ylabel('Sequence Position')
plt.colorbar(im0, ax=axes[0, 0])

# Target activation
im1 = axes[0, 1].imshow(target_act[:64, :64], aspect='auto', cmap='RdBu')
axes[0, 1].set_title('Target Activation (Transformer Output)')
axes[0, 1].set_xlabel('Hidden Dim')
axes[0, 1].set_ylabel('Sequence Position')
plt.colorbar(im1, ax=axes[0, 1])

# NCA prediction
im2 = axes[0, 2].imshow(pred_act[:64, :64], aspect='auto', cmap='RdBu')
axes[0, 2].set_title('NCA Prediction')
axes[0, 2].set_xlabel('Hidden Dim')
axes[0, 2].set_ylabel('Sequence Position')
plt.colorbar(im2, ax=axes[0, 2])

# Error
error = jnp.abs(pred_act - target_act)
im3 = axes[1, 0].imshow(error[:64, :64], aspect='auto', cmap='hot')
axes[1, 0].set_title('Absolute Error')
axes[1, 0].set_xlabel('Hidden Dim')
axes[1, 0].set_ylabel('Sequence Position')
plt.colorbar(im3, ax=axes[1, 0])

# Per-position MSE
pos_mse = jnp.mean(jnp.square(pred_act - target_act), axis=-1)
axes[1, 1].plot(pos_mse)
axes[1, 1].set_title('MSE by Sequence Position')
axes[1, 1].set_xlabel('Sequence Position')
axes[1, 1].set_ylabel('MSE')

# Per-feature MSE
feat_mse = jnp.mean(jnp.square(pred_act - target_act), axis=0)
axes[1, 2].plot(feat_mse[:200])  # First 200 features
axes[1, 2].set_title('MSE by Hidden Dimension (first 200)')
axes[1, 2].set_xlabel('Hidden Dimension')
axes[1, 2].set_ylabel('MSE')

plt.tight_layout()
plt.show()

## Effect of NCA Steps

In [ ]:
# Test how MSE changes with number of NCA steps
step_counts = [8, 16, 32, 64, 96, 128]
step_results = []

for steps in step_counts:
    result = evaluate(nca, test_dataset, steps)
    step_results.append({
        'steps': steps,
        'mse': result['mse'],
        'cosine_sim': result['cosine_similarity'],
    })
    print(f"Steps: {steps:3d} | MSE: {result['mse']:.6f} | Cosine Sim: {result['cosine_similarity']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = [r['steps'] for r in step_results]
mses = [r['mse'] for r in step_results]
cosines = [r['cosine_sim'] for r in step_results]

axes[0].plot(steps, mses, 'o-')
axes[0].set_xlabel('NCA Steps')
axes[0].set_ylabel('MSE')
axes[0].set_title('MSE vs NCA Steps')

axes[1].plot(steps, cosines, 'o-')
axes[1].set_xlabel('NCA Steps')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Cosine Similarity vs NCA Steps')

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("="*60)
print("GPT-2 Block Distillation Summary")
print("="*60)
print(f"\nSource: GPT-2 small, block {block_idx}")
print(f"Target: Neural Cellular Automata")
print(f"\nParameter Comparison:")
print(f"  GPT-2 block:  {gpt2_block_params:,} params")
print(f"  NCA:          {nca_params:,} params")
print(f"  Compression:  {gpt2_block_params / nca_params:.1f}x")
print(f"\nNCA Configuration:")
print(f"  Grid size:    {seq_len} x {compressed_dim} (compressed from {hidden_dim})")
print(f"  Channels:     {channel_size}")
print(f"  Perception:   {perception_size}")
print(f"  Hidden:       {hidden_layer_sizes}")
print(f"  Steps:        {num_nca_steps}")
print(f"\nFinal Test Results:")
print(f"  MSE:          {results['mse']:.6f}")
print(f"  Cosine Sim:   {results['cosine_similarity']:.4f}")
print("="*60)